In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler


# 1. Define file paths and settings
input_path = Path(
    "monthly_records.csv"
)

output_folder = Path(
    "results/step_04"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

criteria = [
    "Average_Quantity",
    "Volatility",
    "Average_Unit_Margin",
    "Total_Cost"
]

FUZZY_MARGIN = 0.05

# Use the same convention as in Code 3.
CLIP_FUZZY_BOUNDS = False


# 2. Define the TOPSIS scoring function
def calculate_topsis(values, weights):

    weighted_values = (
        np.asarray(values, dtype=float)
        * np.asarray(weights, dtype=float)
    )

    positive_ideal = weighted_values.max(axis=0)
    negative_ideal = weighted_values.min(axis=0)

    distance_positive = np.linalg.norm(
        weighted_values - positive_ideal,
        axis=1
    )

    distance_negative = np.linalg.norm(
        weighted_values - negative_ideal,
        axis=1
    )

    total_distance = (
        distance_positive + distance_negative
    )

    return np.divide(
        distance_negative,
        total_distance,
        out=np.full(len(weighted_values), 0.5),
        where=total_distance > 1e-12
    )


# 3. Define the historical MCDM calculation
def calculate_historical_scores(
    history,
    clip_fuzzy=False
):

    # Aggregate only observations available at the current date.
    aggregated = (
        history
        .groupby("Reference", sort=True)
        .agg(
            Average_Quantity=("Quantity", "mean"),
            Volatility=("Quantity", "std"),
            Average_Unit_Margin=("Unit_Margin", "mean"),
            Total_Cost=("Total_Purchase_Cost", "sum")
        )
        .reset_index()
    )

    aggregated["Volatility"] = (
        aggregated["Volatility"].fillna(0)
    )

    if len(aggregated) < 2:
        raise ValueError(
            "At least two parts are required."
        )

    # Normalize the criteria using the available history.
    normalized = MinMaxScaler().fit_transform(
        aggregated[criteria]
    )

    # Reverse volatility: lower variability receives higher priority.
    normalized[:, 1] = (
        1 - normalized[:, 1]
    )

    # Calculate robust Pearson correlations.
    centered = (
        normalized - normalized.mean(axis=0)
    )

    norms = np.linalg.norm(
        centered,
        axis=0
    )

    denominator = np.outer(
        norms,
        norms
    )

    correlation = np.divide(
        centered.T @ centered,
        denominator,
        out=np.zeros((len(criteria), len(criteria))),
        where=denominator > 1e-12
    )

    np.fill_diagonal(
        correlation,
        1.0
    )

    # Recalculate CRITIC weights at the current date.
    information = (
        normalized.std(axis=0, ddof=1)
        * (1 - correlation).sum(axis=1)
    )

    if information.sum() > 1e-12:
        weights = (
            information / information.sum()
        )

    else:
        weights = (
            np.ones(len(criteria)) / len(criteria)
        )

    # Construct and defuzzify triangular fuzzy numbers.
    fuzzy_values = np.stack(
        [
            normalized - FUZZY_MARGIN,
            normalized,
            normalized + FUZZY_MARGIN
        ],
        axis=-1
    )

    if clip_fuzzy:
        fuzzy_values = np.clip(
            fuzzy_values,
            0.0,
            1.0
        )

    centroid_values = fuzzy_values.mean(
        axis=-1
    )

    # Apply the centroid-based TOPSIS procedure.
    aggregated["Fuzzy_TOPSIS_Score"] = (
        calculate_topsis(
            centroid_values,
            weights
        )
    )

    score_table = aggregated[
        [
            "Reference",
            "Fuzzy_TOPSIS_Score"
        ]
    ].copy()

    return score_table, weights


# 4. Load the monthly records
df = pd.read_csv(
    input_path,
    parse_dates=["Entry_Date"]
)

required_columns = [
    "Reference",
    "Entry_Date",
    "Quantity",
    "Unit_Margin",
    "Total_Purchase_Cost"
]

missing_columns = (
    set(required_columns) - set(df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

if df.duplicated(
    ["Reference", "Entry_Date"]
).any():
    raise ValueError(
        "Duplicate part-month observations were found."
    )

numeric_columns = [
    "Quantity",
    "Unit_Margin",
    "Total_Purchase_Cost"
]

if not np.isfinite(
    df[numeric_columns].to_numpy(dtype=float)
).all():
    raise ValueError(
        "The input contains missing or infinite numeric values."
    )

df = df.sort_values(
    ["Entry_Date", "Reference"]
).reset_index(drop=True)


# 5. Verify that the monthly panel is complete
dates = pd.DatetimeIndex(
    sorted(df["Entry_Date"].unique())
)

expected_dates = pd.date_range(
    start=dates.min(),
    end=dates.max(),
    freq="MS"
)

if not dates.equals(expected_dates):
    raise ValueError(
        "Dates must be consecutive month-start dates."
    )

n_parts = df["Reference"].nunique()

if len(df) != len(dates) * n_parts:
    raise ValueError(
        "The panel must contain one observation per part and month."
    )


# 6. Create the monthly time index
date_to_index = {
    date: index
    for index, date in enumerate(dates)
}

df["time_idx"] = (
    df["Entry_Date"]
    .map(date_to_index)
    .astype(int)
)


# 7. Calculate scores separately at each monthly origin
monthly_score_tables = []
monthly_weight_records = []

for time_index, current_date in enumerate(dates):

    # Future observations are excluded from the calculation.
    history = df.loc[
        df["Entry_Date"] <= current_date
    ].copy()

    score_table, current_weights = (
        calculate_historical_scores(
            history,
            clip_fuzzy=CLIP_FUZZY_BOUNDS
        )
    )

    score_table["time_idx"] = time_index

    monthly_score_tables.append(
        score_table
    )

    for criterion, weight in zip(
        criteria,
        current_weights
    ):
        monthly_weight_records.append({
            "Entry_Date": current_date,
            "time_idx": time_index,
            "Criterion": criterion,
            "Weight": float(weight)
        })

    if (
        (time_index + 1) % 12 == 0
        or time_index == len(dates) - 1
    ):
        print(
            f"Processed {time_index + 1}/{len(dates)} months"
        )


# 8. Merge the monthly scores into the original panel
monthly_scores = pd.concat(
    monthly_score_tables,
    ignore_index=True
)

engineered_panel = df.merge(
    monthly_scores,
    on=["Reference", "time_idx"],
    how="left",
    validate="one_to_one"
)

engineered_panel = engineered_panel.sort_values(
    ["time_idx", "Reference"]
).reset_index(drop=True)


# 9. Add cyclical calendar features
# These variables represent the annual seasonal cycle.
engineered_panel["month_sin"] = np.sin(
    2 * np.pi
    * engineered_panel["Entry_Date"].dt.month
    / 12
)

engineered_panel["month_cos"] = np.cos(
    2 * np.pi
    * engineered_panel["Entry_Date"].dt.month
    / 12
)


# 10. Record the chronological data partitions
# Split labels are saved separately from the model input panel.
split_table = pd.DataFrame({
    "Entry_Date": dates,
    "time_idx": np.arange(len(dates))
})

split_table["Partition"] = np.select(
    [
        split_table["Entry_Date"]
        <= pd.Timestamp("2023-12-01"),

        split_table["Entry_Date"]
        <= pd.Timestamp("2024-06-01")
    ],
    [
        "Training",
        "Validation"
    ],
    default="Test"
)


# 11. Check the engineered panel
assert len(engineered_panel) == len(df)

assert not engineered_panel.duplicated(
    ["Reference", "time_idx"]
).any()

assert engineered_panel[
    "Fuzzy_TOPSIS_Score"
].notna().all()

assert engineered_panel[
    "Fuzzy_TOPSIS_Score"
].between(0, 1).all()


# 12. Save the engineered panel and monthly weights
panel_path = (
    output_folder / "engineered_panel.csv"
)

engineered_panel.to_csv(
    panel_path,
    index=False
)

monthly_weights = pd.DataFrame(
    monthly_weight_records
)

monthly_weights.to_csv(
    output_folder / "monthly_critic_weights.csv",
    index=False
)

split_table.to_csv(
    output_folder / "temporal_partitions.csv",
    index=False
)


# 13. Save preprocessing metadata
metadata = {
    "input_file": str(input_path),
    "output_file": str(panel_path),
    "n_rows": len(engineered_panel),
    "n_parts": n_parts,
    "n_months": len(dates),
    "criteria": criteria,
    "weight_method": "CRITIC recalculated at each month",
    "fuzzy_margin": FUZZY_MARGIN,
    "clip_fuzzy_bounds": CLIP_FUZZY_BOUNDS,
    "score_information": (
        "Each monthly score uses observations "
        "up to and including that month only."
    ),
    "forecast_horizon": 6,
    "training_target_end": "2023-12-01",
    "validation_origin": "2023-12-01",
    "test_origin": "2024-06-01"
}

(
    output_folder / "feature_engineering_metadata.json"
).write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)


# 14. Display the results
print("\nEngineered panel created successfully.")
print(f"Output file: {panel_path}")
print(f"Number of rows: {len(engineered_panel):,}")
print(f"Number of columns: {len(engineered_panel.columns)}")

display_columns = [
    "Reference",
    "Entry_Date",
    "Quantity",
    "Unit_Margin",
    "Total_Purchase_Cost",
    "Fuzzy_TOPSIS_Score",
    "time_idx",
    "month_sin",
    "month_cos"
]

print("\nFirst observations:")
print(
    engineered_panel[display_columns]
    .head(10)
    .to_string(index=False)
)

print("\nTemporal partitions:")
print(
    split_table.groupby("Partition")["Entry_Date"]
    .agg(["min", "max", "count"])
)